#จัดการพิกัดและมุมมอง (ROI / Crop / Resize / Perspective)

**หัวข้อหลัก 4/4 ของ "ภาพรวม Image Processing"**

เลือกเฉพาะพื้นที่ที่สนใจ (**ROI/Crop**), เปลี่ยนขนาดภาพ (**Resize**) และแก้ภาพมุมเอียงให้ตรง
ด้วย **Affine / Perspective Transform**  — แต่ละหัวข้อโหลดภาพของตัวเองตอนเริ่ม ปิดท้ายด้วยการหา
4 มุมและ Warp ป้าย FLOWSERVE ที่เอียงจริงให้ตรง

> เนื้อหาและภาพตัวอย่างอ้างอิงจากสไลด์ **COMPUTER VISIONS** โดย Asst.Prof.Dr. Amnach Khawne, King Mongkut's Institute of Technology Ladkrabang


## วิธีใช้ Notebook นี้

- รันทีละ Cell จากบนลงล่างด้วย **Shift + Enter**
- แต่ละหัวข้อแบ่งเป็น Cell ย่อยหลาย Cell ทำทีละขั้นตอน เพื่อให้เห็นผลลัพธ์ทันทีทีละ Cell
- ลองแก้ค่าตัวเลข (parameter) แล้วรันซ้ำ เพื่อดูว่าภาพเปลี่ยนไปอย่างไร
- **ต้องอัปโหลด `data.zip` ก่อน** (Cell ที่ 2) ทุกครั้งที่เปิด Notebook ใหม่ — Colab ลบไฟล์ทิ้งเมื่อ Runtime ถูกรีเซ็ต


## 0. เตรียมเครื่องมือ (Setup)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # แสดงภาพใน Colab แทน cv2.imshow()

print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)


In [ ]:
def show_images(images, titles, cmap=None, figsize=(15, 5)):
    """แสดงภาพหลายภาพเรียงกันในแถวเดียว สำหรับเปรียบเทียบก่อน-หลัง"""
    n = len(images)
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(1, n, i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # BGR -> RGB สำหรับ matplotlib
        plt.title(title, fontsize=11)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 1. เตรียมชุดข้อมูล (Dataset) — อัปโหลด `data.zip`

อัปโหลดไฟล์ **`data.zip`** ที่ได้รับจากผู้สอน (ภาพชุดเดียวกับที่ใช้ในสไลด์ COMPUTER VISIONS)
เมื่อรัน Cell ด้านล่างจะมีปุ่มให้เลือกไฟล์จากเครื่อง — เลือก `data.zip` แล้วรอจนแตกไฟล์เสร็จ

In [ ]:
import os
import zipfile

from google.colab import files

%cd /content
print("เลือกไฟล์ data.zip (ชุดภาพตัวอย่างจากสไลด์ COMPUTER VISIONS)")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall("/content/")

DATA_DIR = "/content/data"
print(f"แตกไฟล์ {zip_filename} เรียบร้อยแล้ว")
print("ไฟล์ภาพที่มีในโฟลเดอร์ data/:")
for fname in sorted(os.listdir(DATA_DIR)):
    print(" -", fname)


## 2. ROI และ Crop — NumPy Slicing (สไลด์ 48)

ภาพที่ใช้: **flower.jpg** — **ROI (Region of Interest)** คือพื้นที่ในภาพที่สนใจ เลือกด้วย `img[y1:y2, x1:x2]`

In [ ]:
img_flower = cv2.imread(f"{DATA_DIR}/flower.jpg")
cv2_imshow(img_flower)


In [ ]:
x1, y1, x2, y2 = 32, 153, 242, 305   # กรอบสี่เหลี่ยมครอบดอกไม้

img_with_box = img_flower.copy()
cv2.rectangle(img_with_box, (x1, y1), (x2, y2), (0, 0, 255), 3)
cv2_imshow(img_with_box)


In [ ]:
roi = img_flower[y1:y2, x1:x2]
print("ขนาด ROI:", roi.shape[1], "x", roi.shape[0])
cv2_imshow(roi)


## 3. Resize — `cv2.resize()`

- รักษาสัดส่วนเดิม: กำหนด `fx`, `fy` เท่ากัน
- ไม่รักษาสัดส่วน: กำหนด `(width, height)` ตรง ๆ

In [ ]:
print(f"1. ภาพต้นฉบับ (ขนาด Width x Height: {roi.shape[1]} x {roi.shape[0]} พิกเซล)")
cv2_imshow(roi)

In [ ]:
# cv2.INTER_AREA เป็นเทคนิคการเกลี่ยพิกเซลที่เหมาะกับ "การย่อรูป" มากที่สุด ทำให้ภาพยังดูคมชัด
resized_keep_ratio = cv2.resize(roi, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)

print(f"2. ย่อแบบคงสัดส่วน 50% (ขนาดใหม่: {resized_keep_ratio.shape[1]} x {resized_keep_ratio.shape[0]} พิกเซล)")
cv2_imshow(resized_keep_ratio)

In [ ]:
# ถ้าสัดส่วน (300, 150) ไม่ตรงกับภาพต้นฉบับ ภาพที่ได้จะดู "ยืด" หรือ "บีบแบน" ผิดธรรมชาติ
resized_stretch = cv2.resize(roi, (300, 150), interpolation=cv2.INTER_LINEAR)

print("3. ปรับขนาดแบบบังคับตายตัว (ขนาดใหม่: 300 x 150 พิกเซล)")
cv2_imshow(resized_stretch)

## 4. เลือก Interpolation ให้เหมาะกับงาน

ภาพที่ใช้: **mask_staircase.png** — Mask รูปขั้นบันได
- **`cv2.INTER_NEAREST`** เหมาะกับ Mask/Class ID (ค่าต้องเดิมเป๊ะ ห้ามผสมค่า)
- **`cv2.INTER_LINEAR`** เหมาะกับภาพทั่วไป แต่ทำให้ขอบ Mask เกิดสีเทาผสม

In [ ]:
img_stair = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/mask_staircase.png"), cv2.COLOR_BGR2GRAY)
cv2_imshow(img_stair)


In [ ]:
# INTER_NEAREST คือการขยายโดยการ "ก็อปปี้ค่าสีพิกเซลที่อยู่ใกล้ที่สุด"
# ทำให้สีดำ(0) ก็ยังเป็นดำ สีขาว(255) ก็ยังเป็นขาว ขอบขั้นบันไดจะยังคมกริบเหมือนเดิม
mask_nearest = cv2.resize(img_stair, (400, 400), interpolation=cv2.INTER_NEAREST)

print("2. ขยายแบบ NEAREST (ขอบคมกริบ สีไม่เพี้ยน)")
cv2_imshow(mask_nearest)

In [ ]:
# INTER_LINEAR คือการขยายโดยการ "คำนวณค่าเฉลี่ยของสีรอบๆ" ให้เนียนขึ้น
# แต่วิธีนี้จะทำให้รอยต่อระหว่างสีดำและสีขาว ถูกเกลี่ยจนกลายเป็น "สีเทา" (Gray Fringe)
mask_linear = cv2.resize(img_stair, (400, 400), interpolation=cv2.INTER_LINEAR)

print("3. ขยายแบบ LINEAR (ขอบเบลอ เกิดเป็นสีเทาแทรก)")
cv2_imshow(mask_linear)

In [ ]:
import numpy as np

# ตรวจสอบว่าในภาพ mask_linear มีกี่พิกเซลที่ไม่ใช่สีดำ (0) และไม่ใช่สีขาว (255)
# ซึ่งก็คือ "สีเทา" ที่เกิดจากการเกลี่ยสีนั่นเอง
gray_fringe = np.sum((mask_linear != 0) & (mask_linear != 255))

print(f"จำนวนพิกเซลที่เป็นสีเทาผสม (ไม่ใช่ 0 หรือ 255) หลังใช้ LINEAR: {gray_fringe} พิกเซล")
print("สรุป: INTER_LINEAR ไม่เหมาะกับการ Resize ภาพประเภท Mask หรือ Label อย่างยิ่ง!")

## 5. Affine Transform — เส้นขนานยังคงขนาน

ภาพที่ใช้: **label_tilted.jpg** — ป้าย FLOWSERVE ใบเดียวกับที่ใช้ตลอดหัวข้อ Affine/Perspective ในสไลด์
ใช้จุดอ้างอิง 3 จุดที่ไม่อยู่แนวเส้นเดียวกัน คู่กับ `cv2.warpAffine()`

In [ ]:
img_label = cv2.imread(f"{DATA_DIR}/label_tilted.jpg")

print("1. ภาพต้นฉบับ (Original)")
cv2_imshow(img_label)

In [ ]:
h, w = img_label.shape[:2]
center = (w / 2, h / 2)

# angle=15 คือหมุนทวนเข็มนาฬิกา 15 องศา, scale=1.0 คือขนาดเท่าเดิมไม่ซูมเข้าซูมออก
M_rotate = cv2.getRotationMatrix2D(center, angle=15, scale=1.0)

# borderValue=(0, 0, 0) คือการเติมสีดำในพื้นที่ขอบที่เกิดจากการหมุน
rotated = cv2.warpAffine(img_label, M_rotate, (w, h), borderValue=(0, 0, 0))

print("2. ผลลัพธ์การหมุนภาพ 15 องศา (Rotated 15 deg)")
cv2_imshow(rotated)

In [ ]:
# การทำ Affine Transform คือการกำหนด "จุดอ้างอิง 3 จุด" ในภาพเดิม
# แล้วสั่งให้ย้ายไปยัง "พิกัดใหม่ 3 จุด" ภาพทั้งภาพจะถูกดึงและบิดตามจุดเหล่านั้น
src_pts = np.float32([[50, 50], [300, 50], [50, 300]])
dst_pts = np.float32([[10, 150], [300, 50], [150, 350]])

M_shear = cv2.getAffineTransform(src_pts, dst_pts)
sheared = cv2.warpAffine(img_label, M_shear, (w, h), borderValue=(0, 0, 0))

print("3. ผลลัพธ์การบิดภาพด้วยการย้ายจุด 3 จุด (Sheared - 3 pts)")
cv2_imshow(sheared)

## 6. Perspective Transform บนภาพจริง

ใช้ **img_label ภาพเดิมจากหัวข้อที่แล้ว** — ขั้นแรก: threshold แยกป้ายออกจากพื้นหลัง แล้วหาเส้นขอบ

In [ ]:
cv2_imshow(img_label)

In [ ]:
gray_label = cv2.cvtColor(img_label, cv2.COLOR_BGR2GRAY)

print("2. ภาพหลังแปลงเป็นขาวดำ (Grayscale)")
cv2_imshow(gray_label)

In [ ]:
# ใช้ Threshold แบ่งความสว่าง โดยตั้งเกณฑ์ (T) ไว้ที่ 60
# จุดที่สว่างกว่า 60 จะกลายเป็นสีขาว (255) จุดที่มืดกว่าจะกลายเป็นสีดำ (0)
_, label_th = cv2.threshold(gray_label, 60, 255, cv2.THRESH_BINARY)

print("3. ภาพหลังทำ Threshold (T=60)")
cv2_imshow(label_th)

### หา 4 มุมของป้ายด้วย `cv2.findContours()` + `cv2.approxPolyDP()`

In [ ]:
contours, _ = cv2.findContours(label_th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# ในภาพอาจมีฝุ่นหรือเส้นขอบเล็ก ๆ ติดมาด้วย จึงใช้ max() เพื่อเลือกเฉพาะเส้นขอบที่มี
# "พื้นที่ (Area) ใหญ่ที่สุด" ซึ่งก็คือตัวแผ่นป้าย/ฉลากที่ต้องการ
label_contour = max(contours, key=cv2.contourArea)

print("1. ค้นพบเส้นขอบที่ใหญ่ที่สุดเรียบร้อยแล้ว (แผ่นป้าย)")

In [ ]:
import numpy as np

peri = cv2.arcLength(label_contour, True)

# approxPolyDP: เป็นคำสั่ง "ลดความซับซ้อนของเส้น" (Approximation)
# โดยจะพยายามดัดเส้นโค้งหยัก ๆ ให้กลายเป็นเส้นตรง
# 0.02 * peri คือความแม่นยำ (ถ้าปรับตัวเลขนี้ให้มากขึ้น จะยิ่งตัดมุมยิบย่อยออกไปเยอะ)
approx = cv2.approxPolyDP(label_contour, 0.02 * peri, True).reshape(-1, 2).astype(np.float32)

print(f"2. จำนวนมุมที่คอมพิวเตอร์หาได้: {len(approx)} มุม")

In [ ]:
corners_vis = img_label.copy()
cv2.drawContours(corners_vis, [approx.astype(int)], -1, (0, 0, 255), 2)  # เส้นขอบสีแดง

for pt in approx:
    cv2.circle(corners_vis, tuple(pt.astype(int)), 6, (255, 0, 0), -1)   # จุดมุมสีน้ำเงิน

print("3. ผลลัพธ์: การค้นหาและมาร์คจุดมุมทั้ง 4 ของแผ่นป้าย")
cv2_imshow(corners_vis)

### เรียงมุม TL/TR/BR/BL แล้ว Warp ให้ตรง

ต้องเรียง 4 จุดที่หาได้เป็นลำดับ **TL → TR → BR → BL** เสมอ ก่อนส่งเข้า `cv2.getPerspectiveTransform()`
ใช้เทคนิค "ผลบวก/ผลต่าง" ของพิกัด (x+y และ y−x) ในการเรียงมุมโดยไม่ต้องไล่ดูเอง

In [ ]:
import numpy as np

# คอมพิวเตอร์เจอมุม 4 มุมแล้ว แต่มันไม่รู้ว่ามุมไหนคือ ซ้ายบน ขวาบน ขวาล่าง ซ้ายล่าง
# เราจึงต้องเขียนฟังก์ชัน (Function) เพื่อใช้คณิตศาสตร์มาช่วยเรียงลำดับให้ถูกต้องเสมอ
def order_points(pts):
    # s (x+y): มุมซ้ายบนสุด x,y จะน้อยสุด (min) / มุมขวาล่างสุด x,y จะมากสุด (max)
    s = pts.sum(axis=1)
    tl = pts[np.argmin(s)] # ซ้ายบน (Top-Left)
    br = pts[np.argmax(s)] # ขวาล่าง (Bottom-Right)

    # diff (y-x): มุมขวาบนสุด y จะน้อย x จะมาก (min) / มุมซ้ายล่างสุด y จะมาก x จะน้อย (max)
    diff = np.diff(pts, axis=1).flatten()
    tr = pts[np.argmin(diff)] # ขวาบน (Top-Right)
    bl = pts[np.argmax(diff)] # ซ้ายล่าง (Bottom-Left)

    # ส่งคืนค่ามุมที่เรียงลำดับแล้ว (ซ้ายบน, ขวาบน, ขวาล่าง, ซ้ายล่าง)
    return np.array([tl, tr, br, bl], dtype=np.float32)

# นำมุมทั้ง 4 ที่ได้จากขั้นตอนที่แล้ว (approx) มาเรียงลำดับ
ordered = order_points(approx)
print("พิกัดมุมที่ถูกเรียงลำดับแล้ว (TL, TR, BR, BL):")
print(ordered)

In [ ]:
# แยกพิกัดที่เรียงแล้วออกมาเก็บในตัวแปร 4 ตัว
(tl, tr, br, bl) = ordered

# คำนวณความกว้าง (Width): หาระยะห่างระหว่าง ซ้ายล่าง-ขวาล่าง และ ซ้ายบน-ขวาบน แล้วเลือกอันที่ยาวที่สุด
# (np.linalg.norm คือสูตรคณิตศาสตร์ที่ใช้หาระยะห่างระหว่างจุด 2 จุด)
width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))

# คำนวณความสูง (Height): หาระยะห่างระหว่าง ขวาบน-ขวาล่าง และ ซ้ายบน-ซ้ายล่าง แล้วเลือกอันที่ยาวที่สุด
height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))

print(f"ขนาดของแผ่นป้ายที่เราจะดึงให้ตรงคือ: กว้าง {width} x สูง {height} พิกเซล")

In [ ]:
# กำหนดพิกัดของ "กรอบสี่เหลี่ยมผืนผ้าตรง ๆ" ที่ต้องการนำภาพไปแปะ เริ่มจาก (0,0)
# ไปจนถึงความกว้างและความสูงที่คำนวณไว้
dst_rect = np.float32([
    [0, 0],
    [width - 1, 0],
    [width - 1, height - 1],
    [0, height - 1]
])

# สร้างสมการแปลงมุมมอง: จับคู่จุดมุมที่เอียงอยู่ (ordered) เข้ากับกรอบตรง ๆ (dst_rect)
M_perspective = cv2.getPerspectiveTransform(ordered, dst_rect)
straightened = cv2.warpPerspective(img_label, M_perspective, (width, height))

print("ภาพหลังจากการดึงให้ตรง (Perspective Transform)")
cv2_imshow(straightened)

## สรุปสิ่งที่เรียนในบทนี้

| แนวคิด | คำสั่งที่ใช้ | Parameter สำคัญ |
|---|---|---|
| เลือกพื้นที่ (ROI/Crop) | NumPy slicing `img[y1:y2, x1:x2]` | พิกัด x1,y1,x2,y2 |
| เปลี่ยนขนาด | `cv2.resize()` | fx, fy หรือ (width, height) |
| เลือก Interpolation | `cv2.INTER_NEAREST/LINEAR` | ชนิดข้อมูล (ภาพ vs Mask) |
| หมุน/เฉือน (Affine) | `cv2.getRotationMatrix2D()`, `cv2.getAffineTransform()`, `cv2.warpAffine()` | มุม/จุดอ้างอิง 3 คู่ |
| หาขอบวัตถุ | `cv2.threshold()`, `cv2.findContours()` | T |
| หา 4 มุม | `cv2.approxPolyDP()` | epsilon |
| แก้มุมมอง (Perspective) | `cv2.getPerspectiveTransform()`, `cv2.warpPerspective()` | จุดอ้างอิง 4 คู่ (TL/TR/BR/BL) |

จบทั้ง 4 หัวข้อของ **"ภาพรวม Image Processing"** — นำเทคนิคทั้ง 4 บทไปประยุกต์กับงานตรวจสอบชิ้นงานจริงในโรงงานได้
(หัวข้อ Object Detection ด้วย YOLO26 เป็นเนื้อหาลำดับถัดไปของหลักสูตร ไม่ได้รวมอยู่ใน Notebook ชุดนี้)


## แบบฝึกหัดท้ายบท (ลองทำเอง)

1. เปลี่ยนพิกัด ROI ในหัวข้อ 2 ให้ครอบส่วนอื่นของภาพดอกไม้
2. เปลี่ยนมุมหมุนในหัวข้อ 5 จาก 15 เป็น 45 องศา
3. เปลี่ยนค่า threshold `60` ในหัวข้อ 6 เป็นค่าอื่น แล้วดูว่ายังหาป้ายเจอ 4 มุมครบหรือไม่
4. นำ `label_th` ไปหา contour ด้วย `cv2.RETR_LIST` แทน `cv2.RETR_EXTERNAL` แล้วสังเกตความแตกต่าง
